In [20]:
import sys

sys.path.append("../../")


In [21]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

from models.base import Base
from models.boulder import Boulder
from models.similarity import Similarity



db_path = "matchit.db"

DATABASE_URL = f"sqlite:///../../{db_path}"
engine = create_engine(DATABASE_URL, echo=False)

Base.metadata.create_all(engine)


session = Session(bind=engine)

In [36]:
from sqlalchemy import func

from models.grade import Grade


text = "inesc"

boulder_search = session.execute(
    select(Boulder.id, Boulder.name, Grade.value, func.count(Boulder.ascents))
    .join(Boulder.grade)
    .join(Boulder.ascents)
    .where(Boulder.name_normalized.like(f"%{text}%"))
    .group_by(Boulder.id)
).all()
boulder_search

[(10852, 'Ineschakra', '7B+', 312),
 (10853, 'Ineschakra de Pie', '6B', 1),
 (10854, 'Ineschakra stand', '7A', 27),
 (10874, 'Inésch-Manuch', '7C', 1)]

In [37]:
import time


boulder_id = 10852

boulder = session.get(Boulder, boulder_id)
print("Selected boulder:")
print(boulder)

print("\nSimilar boulders:")
start = time.time()
similarities = session.execute(
    select(Boulder, Similarity.score)
    .join(Similarity, Boulder.id == Similarity.id2)
    .where(Similarity.id1 == boulder_id)
    .order_by(Similarity.score.desc())
    .limit(10)
).all()
end = time.time()
print(f"Query took {end - start:.4f} seconds")
for boulder, score in similarities:
    print(boulder, score)

Selected boulder:
<Boulder(name: Ineschakra, grade: 7B+, Ascents: 312)>

Similar boulders:
Query took 0.0165 seconds
<Boulder(name: Manuchakra, grade: 7C, Ascents: 395)> 1.6824196577072144
<Boulder(name: Rammstein, grade: 7A+, Ascents: 758)> 0.944444477558136
<Boulder(name: Palpant, grade: 7B+, Ascents: 321)> 0.9158878922462463
<Boulder(name: Aerolineas, grade: 7B, Ascents: 335)> 0.9032846689224243
<Boulder(name: A Ciegas, grade: 7A+, Ascents: 543)> 0.8722527027130127
<Boulder(name: Eclipse, grade: 7B, Ascents: 752)> 0.7952069640159607
<Boulder(name: La trave de techos, grade: 7B, Ascents: 243)> 0.7933194041252136
<Boulder(name: El Varano, grade: 7C, Ascents: 603)> 0.7692307829856873
<Boulder(name: Dr. Zoiberg, grade: 7A+, Ascents: 387)> 0.738916277885437
<Boulder(name: Totem aka Plus del Autobús, grade: 7A+, Ascents: 445)> 0.7175226211547852


In [24]:
boulder_id = 3787

boulder = session.get(Boulder, boulder_id)

print(f"{'='*60}")
print(f"Boulder: {boulder.name}")
print(f"{'='*60}")
print(f"ID:              {boulder.id}")
print(f"Grade:           {boulder.grade.value if boulder.grade else 'N/A'}")
print(f"Normalized Name: {boulder.name_normalized}")
print(f"Total Ascents:   {len(boulder.ascents)}")
print(f"Crag:            {boulder.crag.name if boulder.crag else 'N/A'}")
print(f"Area:            {boulder.crag.area.name if boulder.crag and boulder.crag.area else 'N/A'}")
print(f"Similarity idx:  {boulder.similarity_matrix_id if boulder.similarity_matrix_id is not None else 'N/A'}")
print(f"\n{'Ascents Details:':-^60}")
for i, ascent in enumerate(boulder.ascents[:10], 1):
    print(f"  {i}. User ID: {ascent.user_id}, Date: {ascent.log_date if hasattr(ascent, 'log_date') else 'N/A'}")
if len(boulder.ascents) > 10:
    print(f"  ... and {len(boulder.ascents) - 10} more ascents")
print(f"{'='*60}")

Boulder: Vecchio Leone
ID:              3787
Grade:           8B
Normalized Name: vecchio leone
Total Ascents:   101
Crag:            Brione
Area:            Ticino
Similarity idx:  6472

----------------------Ascents Details:----------------------
  1. User ID: 3446, Date: 2025-11-26
  2. User ID: 2800, Date: 2025-11-20
  3. User ID: 2278, Date: 2025-11-13
  4. User ID: 3106, Date: 2025-11-07
  5. User ID: 2775, Date: 2025-11-05
  6. User ID: 1395, Date: 2025-10-29
  7. User ID: 3190, Date: 2025-10-25
  8. User ID: 392, Date: 2025-10-25
  9. User ID: 3187, Date: 2025-05-16
  10. User ID: 2782, Date: 2025-03-27
  ... and 91 more ascents


In [25]:
from ml.crud import delete_similarity_data


# delete_similarity_data(session=session, area_slug="")

In [26]:
similarities = session.scalar(select(func.count(Similarity.id1)))
print(f"Total similarity entries: {similarities}")

Total similarity entries: 3439836


In [33]:
import time
from sqlalchemy import text, func
from models.similarity import Similarity

# Test boulder IDs (use actual IDs from your database)
boulder_ids = [1, 100, 500, 1000, 5000]

# Method 1: Raw SQL (fastest)
t = time.time()
placeholders = ','.join(f':id{i}' for i in range(len(boulder_ids)))
params = {f'id{i}': boulder_id for i, boulder_id in enumerate(boulder_ids)}
result = session.execute(
    text(f"""
        SELECT id2 as similar_id, SUM(score) as total_score
        FROM similarity
        WHERE id1 IN ({placeholders})
        GROUP BY id2
        ORDER BY total_score DESC
        LIMIT 50
    """),
    params
).fetchall()
print(f"Raw SQL: {(time.time()-t)*1000:.1f}ms, Results: {len(result)}")

# Method 2: ORM query
t = time.time()
result = (
    session.query(
        Similarity.id2.label('similar_id'),
        func.sum(Similarity.score).label('total_score')
    )
    .filter(Similarity.id1.in_(boulder_ids))
    .group_by(Similarity.id2)
    .order_by(func.sum(Similarity.score).desc())
    .limit(50)
    .all()
)
print(f"ORM query: {(time.time()-t)*1000:.1f}ms, Results: {len(result)}")

Raw SQL: 2.9ms, Results: 50
ORM query: 4.8ms, Results: 50
